# Test chi tiết — 20 case: giá & hệ số nhân (thực vs dự đoán)

Notebook train 3 model (giá cơ bản, hệ số nhân, giá trực tiếp) theo tháng, dự đoán trên tập test,
rồi hiển thị **20 chuyến cụ thể** để xem **chênh lệch thực tế** giữa dự đoán và giá thật.

Giá cuối lấy theo **Hướng 2** (giá cơ bản × hệ số nhân) vì đã chứng minh tốt hơn.

**1. Nạp dữ liệu & train 3 model theo tháng**

In [1]:
import warnings, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings("ignore")
pd.set_option("display.width",260); pd.set_option("display.max_columns",40)

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chay chuan_bi_du_lieu.ipynb truoc!"
COLS = ["service_name","pickup_location_name","dropoff_location_name","weather_main",
        "quote_distance","quote_duration","gio_vn","target_hour",
        "latest_observed_price","latest_observed_multiplier","latest_observed_quote_distance","latest_observed_quote_duration",
        "history_60m_price_mean","history_60m_price_std","history_60m_price_slope_per_minute","actual_observation_age_minutes",
        "pricing_market_imbalance_5m_lag","pricing_demand_index_5m_lag","pricing_supply_index_5m_lag","pricing_quote_count_5m_lag",
        "target_shown_price","target_shown_multiplier","evaluation_month","split"]
df = pd.read_parquet(PREP, columns=COLS)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)

CAT=["service_name","pickup_location_name","dropoff_location_name","weather_main"]
D_NUM=["quote_distance","quote_duration","gio_vn","latest_observed_price","history_60m_price_mean",
       "history_60m_price_std","history_60m_price_slope_per_minute","latest_observed_quote_distance",
       "latest_observed_quote_duration","actual_observation_age_minutes"]
B_NUM=["quote_distance","quote_duration","gio_vn","latest_observed_base","history_60m_price_mean",
       "history_60m_price_std","latest_observed_quote_distance","latest_observed_quote_duration"]
M_NUM=["pricing_market_imbalance_5m_lag","pricing_demand_index_5m_lag","pricing_supply_index_5m_lag",
       "pricing_quote_count_5m_lag","latest_observed_multiplier","gio_vn","actual_observation_age_minutes"]
def prep(d,num):
    X=d[CAT+num].copy()
    for c in CAT: X[c]=X[c].astype("category")
    return X
def tao():
    return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, categorical_features=CAT, random_state=42)

res=[]
for th in sorted(df.evaluation_month.unique()):
    sub=df[df.evaluation_month==th]; tr=sub[sub.split=="train"]; te=sub[sub.split=="test"].copy()
    t0=time.time()
    m1=tao().fit(prep(tr,D_NUM), np.log(tr.target_shown_price)); te["gia_H1"]=np.exp(m1.predict(prep(te,D_NUM)))
    mA=tao().fit(prep(tr,B_NUM), np.log(tr.base_price)); base=np.exp(mA.predict(prep(te,B_NUM)))
    mB=tao().fit(prep(tr,M_NUM), tr.target_shown_multiplier); mult=mB.predict(prep(te,M_NUM))
    te["surge_pred"]=mult; te["gia_H2"]=base*mult; te["base_pred"]=base
    res.append(te); print(f"  [{th}] test={len(te):,} | {time.time()-t0:.1f}s")
allte=pd.concat(res).reset_index(drop=True)
print(f"Xong. {len(allte):,} chuyen test.")

  [2026-01] test=315,360 | 56.8s


  [2026-02] test=234,632 | 54.7s


  [2026-03] test=314,368 | 56.2s
Xong. 864,360 chuyen test.


**2. 20 test case chi tiết**

Chọn 20 chuyến (10 CÓ surge + 10 giá thường) để thấy rõ chênh lệch. Cột: tuyến, dịch vụ, giờ VN,
quãng đường, thời lượng, **giá thực / giá dự đoán / sai số**, **hệ số nhân thực / dự đoán / sai số**.

In [2]:
allte["gio_vn"]=(allte.target_hour+7)%24
co = allte[allte.target_shown_multiplier>1.1].sample(10, random_state=7)
khong = allte[allte.target_shown_multiplier<=1.0].sample(10, random_state=7)
case = pd.concat([co,khong]).reset_index(drop=True)

out = pd.DataFrame({
  "Tuyen": case.pickup_location_name.str[:10]+" -> "+case.dropoff_location_name.str[:10],
  "Dich vu": case.service_name.str.replace("Synthetic ","").str.replace(" Car",""),
  "Gio": case.gio_vn.astype(int),
  "Km": case.quote_distance.round(1),
  "Phut": (case.quote_duration/60).round(0).astype(int),
  "GIA THUC(k)": (case.target_shown_price/1000).round(0).astype(int),
  "GIA DUDOAN(k)": (case.gia_H2/1000).round(0).astype(int),
  "Sai so(k)": ((case.gia_H2-case.target_shown_price)/1000).round(1),
  "Sai so%": (abs(case.gia_H2-case.target_shown_price)/case.target_shown_price*100).round(1),
  "SURGE THUC": case.target_shown_multiplier.round(2),
  "SURGE DUDOAN": case.surge_pred.round(2),
  "Sai so surge": (case.surge_pred-case.target_shown_multiplier).round(2),
})
print("20 TEST CASE CHI TIET (10 co surge + 10 gia thuong):")
display(out)

20 TEST CASE CHI TIET (10 co surge + 10 gia thuong):


,Tuyen,Dich vu,Gio,Km,Phut,GIA THUC(k),GIA DUDOAN(k),Sai so(k),Sai so%,SURGE THUC,SURGE DUDOAN,Sai so surge
0,Crescent M -> SC Vivo Ci,Standard,14,5.5,30,172,122,-50.4,29.3,1.22,1.23,0.01
1,Crescent M -> SC Vivo Ci,Premium,15,6.8,24,144,125,-19.1,13.3,1.18,1.18,0.00
2,Crescent M -> SC Vivo Ci,Standard,15,5.7,22,105,101,-4.1,3.9,1.11,1.08,-0.03
3,EcoGreen S -> Crescent M,Standard,12,5.0,20,104,109,4.7,4.5,1.32,1.34,0.02
4,SC Vivo Ci -> Crescent M,Premium,14,6.3,21,139,122,-17.0,12.2,1.29,1.28,-0.01
5,EcoGreen S -> Crescent M,Premium,9,4.9,23,77,107,30.5,39.6,1.29,1.27,-0.02
6,EcoGreen S -> SC Vivo Ci,Standard,18,9.1,58,292,256,-36.4,12.5,1.57,1.57,0.00
7,EcoGreen S -> Crescent M,Standard,19,4.7,29,138,117,-20.8,15.1,1.30,1.31,0.01
8,EcoGreen S -> Crescent M,Premium,12,5.5,41,151,151,-0.1,0.1,1.36,1.37,0.01
9,SC Vivo Ci -> EcoGreen S,Standard,9,7.7,25,144,134,-9.9,6.9,1.27,1.19,-0.08


**3. Đọc nhanh vài case**

In dạng câu chuyện cho 3 chuyến đầu để dễ hình dung.

In [3]:
for i in range(3):
    r=case.iloc[i]
    print(f"--- Chuyen {i+1}: {r.service_name.replace('Synthetic ','')} {r.pickup_location_name} -> {r.dropoff_location_name} ---")
    print(f"   {r.quote_distance:.1f} km, {r.quote_duration/60:.0f} phut, luc {(int(r.target_hour)+7)%24}h VN, thoi tiet {r.weather_main}")
    print(f"   Cung-cau imbalance={r.pricing_market_imbalance_5m_lag:.2f} | surge quan sat gan nhat={r.latest_observed_multiplier:.2f}")
    print(f"   HE SO NHAN : thuc {r.target_shown_multiplier:.2f}  | du doan {r.surge_pred:.2f}")
    print(f"   GIA CUOI   : thuc {int(r.target_shown_price):,}  | du doan {int(r.gia_H2):,} VND  "
          f"(lech {int(r.gia_H2-r.target_shown_price):+,})\n")

--- Chuyen 1: Standard Car Crescent Mall -> SC Vivo City ---
   5.5 km, 30 phut, luc 14h VN, thoi tiet Clouds
   Cung-cau imbalance=1.77 | surge quan sat gan nhat=1.24
   HE SO NHAN : thuc 1.22  | du doan 1.23
   GIA CUOI   : thuc 172,000  | du doan 121,584 VND  (lech -50,415)

--- Chuyen 2: Premium Car Crescent Mall -> SC Vivo City ---
   6.8 km, 24 phut, luc 15h VN, thoi tiet Clouds
   Cung-cau imbalance=1.38 | surge quan sat gan nhat=1.22
   HE SO NHAN : thuc 1.18  | du doan 1.18
   GIA CUOI   : thuc 144,000  | du doan 124,884 VND  (lech -19,115)

--- Chuyen 3: Standard Car Crescent Mall -> SC Vivo City ---
   5.7 km, 22 phut, luc 15h VN, thoi tiet Rain
   Cung-cau imbalance=1.15 | surge quan sat gan nhat=1.07
   HE SO NHAN : thuc 1.11  | du doan 1.08
   GIA CUOI   : thuc 105,000  | du doan 100,937 VND  (lech -4,062)



**4. Tổng hợp độ chính xác trên toàn tập test**

In [4]:
mae_gia = mean_absolute_error(allte.target_shown_price, allte.gia_H2)
mae_surge = mean_absolute_error(allte.target_shown_multiplier, allte.surge_pred)
ae = (allte.gia_H2-allte.target_shown_price).abs()
print(f"GIA  : MAE = {mae_gia:,.0f} VND | MAPE = {(ae/allte.target_shown_price).mean()*100:.1f}%")
print(f"       Ty le chuyen sai < 10k VND: {(ae<10000).mean()*100:.0f}% | <20k: {(ae<20000).mean()*100:.0f}%")
print(f"SURGE: MAE = {mae_surge:.3f} | Ty le sai < 0.1: {((allte.surge_pred-allte.target_shown_multiplier).abs()<0.1).mean()*100:.0f}%")
from sklearn.metrics import roc_auc_score
print(f"       ROC-AUC (co surge) = {roc_auc_score((allte.target_shown_multiplier>1).astype(int), allte.surge_pred):.4f}")

GIA  : MAE = 18,049 VND | MAPE = 14.7%
       Ty le chuyen sai < 10k VND: 38% | <20k: 66%
SURGE: MAE = 0.023 | Ty le sai < 0.1: 98%
       ROC-AUC (co surge) = 0.9979
